In [1]:
import torch
import torch.nn as nn
import json
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import sqlite3
import numpy as np

In [2]:
with open('champ_names.json', 'r') as f:
    champ_names = json.load(f)

conn = sqlite3.connect('league_data.db')
df = pd.read_sql_query("SELECT * FROM matches", conn)

# lots of bugs when case is not set to lower due to discrepancies between api and data dragon
champ_names = [champ.lower() for champ in champ_names]
df[['champ_1', 'champ_2', 'champ_3', 'champ_4', 'champ_5', 'champ_6', 'champ_7', 'champ_8', 'champ_9', 'champ_10']] = df[['champ_1', 'champ_2', 'champ_3', 'champ_4', 'champ_5', 'champ_6', 'champ_7', 'champ_8', 'champ_9', 'champ_10']].apply(lambda x: x.str.lower())

champ_names.append('masked')
conn.close()

In [3]:
print(df.head())

        match_id  champ_1   champ_2   champ_3    champ_4 champ_5      champ_6  \
0  OC1_706346544    vayne    lillia  vladimir     ezreal   karma       aatrox   
1  OC1_706041567  camille   kindred     akali   aphelios  thresh         udyr   
2  OC1_706302834     shen    qiyana     jayce     veigar   senna      drmundo   
3  OC1_706358935     kled     viego       lux  seraphine  zilean  mordekaiser   
4  OC1_706333498    teemo  jarvaniv    qiyana   malzahar    rell     renekton   

   champ_7  champ_8 champ_9  champ_10  
0  naafiri    locke  lucian      sona  
1   graves   xerath  ezreal  nautilus  
2    diana    talon  draven      nami  
3    talon  orianna    jhin  nautilus  
4  warwick   viktor    jinx    velkoz  


In [4]:
champ_data = df.drop(['match_id'], axis=1)

In [5]:
NUM_CHAMPIONS_PER_GAME = 10
NUM_ROLES = 5
EMBEDDING_DIM = 64
HIDDEN_DIM = 512
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
#including masked token
TOTAL_CHAMPIONS = len(champ_names)

BATCH_SIZE = 32

In [6]:
str_to_idx = dict(zip(champ_names, range(len(champ_names))))

In [7]:
encode = lambda name: str_to_idx[name] # takes a champion name and returns the corresponding index
decode = lambda idx: champ_names[idx] # takes index and returns the corresponding champion name

print(decode(encode('monkeyking')))

monkeyking


In [8]:
class ChampionDataset(Dataset):
    def __init__(self, data, encode_fn):
        self.data = data
        self.encode = encode_fn

    def __len__(self):
        return len(self.data)

    # exactly one champion is chosen uniformly at random each game and masked
    def __getitem__(self, idx):
        current_data = self.data.iloc[idx]
        masked_data = current_data.copy()

        masked_id = torch.randint(low=0, high=NUM_CHAMPIONS_PER_GAME, size=(1,)).item()
        masked_champ = masked_data.iat[masked_id]
        masked_data.iloc[masked_id] = 'masked'

        x_df = masked_data.apply(self.encode)
        x_np = x_df.to_numpy(dtype = np.int64)

        return torch.tensor(x_np), torch.tensor(self.encode(masked_champ))

In [9]:
champion_dataset = ChampionDataset(champ_data, encode)

In [10]:
class LeagueDraftModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(num_embeddings=TOTAL_CHAMPIONS, embedding_dim=EMBEDDING_DIM)
        self.role_embedding = nn.Embedding(num_embeddings=NUM_ROLES, embedding_dim=EMBEDDING_DIM)
        self.lm = nn.Linear(EMBEDDING_DIM * NUM_CHAMPIONS_PER_GAME, HIDDEN_DIM)
        self.output_layer = nn.Linear(HIDDEN_DIM, TOTAL_CHAMPIONS)

    def forward(self, x):
        # x is of shape (BATCH_SIZE, NUM_CHAMPIONS_PER_GAME), labels is of shape (BATCH_SIZE)
        batch_size = x.size(0)
        token_embeds = self.token_embedding(x)  # Shape: (BATCH_SIZE, NUM_CHAMPIONS_PER_GAME, EMBEDDING_DIM)
        role_embeds = self.role_embedding(torch.arange(start=0, end=NUM_ROLES).repeat(2)) # Shape: (NUM_CHAMPIONS_PER_GAME, EMBEDDING_DIM)
        embeds = token_embeds + role_embeds # Shape: (BATCH_SIZE, NUM_CHAMPIONS_PER_GAME, EMBEDDING_DIM)

        # flatten
        embeds = embeds.view(batch_size, NUM_CHAMPIONS_PER_GAME * EMBEDDING_DIM) # Shape: (BATCH_SIZE, NUM_CHAMPIONS_PER_GAME * EMBEDDING_DIM)

        pre_activations = torch.tanh(self.lm(embeds))  # Shape: (BATCH_SIZE, HIDDEN_DIM)
        logits = self.output_layer(pre_activations) # Shape: (BATCH_SIZE, TOTAL_CHAMPIONS)
        return logits

In [11]:
train_data, test_data = random_split(champion_dataset, [0.9, 0.1])
print(len(train_data), len(test_data))

10209 1134


In [12]:
train_load = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_load = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

In [13]:
model = LeagueDraftModel()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()

In [14]:
for epoch in range(NUM_EPOCHS):

    running_loss_train = 0.0
    model.train()
    for x_b, y_b in train_load:
        optimizer.zero_grad()
        logits = model(x_b)
        output = loss_fn(logits, y_b)
        output.backward()
        optimizer.step()
        running_loss_train += output.item()
    train_loss = running_loss_train/len(train_load)

    running_loss_test = 0.0
    with torch.no_grad():
        model.eval()
        for x_t, y_t in test_load:
            logits = model(x_t)
            output = loss_fn(logits, y_t)
            running_loss_test += output.item()
        test_loss = running_loss_test/len(test_load)

    print(f'Epoch: {epoch+1}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')


Epoch: 1, Train Loss: 4.4025, Test Loss: 4.0818
Epoch: 2, Train Loss: 4.1491, Test Loss: 4.1885
Epoch: 3, Train Loss: 4.0972, Test Loss: 4.1759
Epoch: 4, Train Loss: 4.0786, Test Loss: 4.0993
Epoch: 5, Train Loss: 4.0755, Test Loss: 4.1127
Epoch: 6, Train Loss: 4.0465, Test Loss: 4.0773
Epoch: 7, Train Loss: 4.0069, Test Loss: 4.0470
Epoch: 8, Train Loss: 4.0082, Test Loss: 4.1392
Epoch: 9, Train Loss: 3.9702, Test Loss: 4.0966
Epoch: 10, Train Loss: 3.9439, Test Loss: 4.0856
